# Qwen3.5 9B — QLoRA Fine-Tuning para Verilog

**Modelo:** `Qwen/Qwen3.5-9B` (4-bit QLoRA en T4 15 GB)  
**Especialidad:** interfaces de comunicación (AXI, UART, SPI, I2C) + creación/modificación de archivos `.v`/`.sv`  
**Benchmark:** VerilogEval + RTLLM + compilación real con iverilog  
**Export:** GGUF Q8_0 listo para LM Studio

> **Nota sobre "full fine-tuning":**  
> El 9B en bfloat16 pesa ~18 GB — no cabe en T4 (15 GB).  
> Usamos **QLoRA** (4-bit) que lo reduce a ~6 GB, permitiendo entrenar  
> LoRA en todas las capas. Para full fine-tuning real del 9B necesitas A100 40 GB (Colab Pro+).

### Antes de empezar
1. `Runtime → Change runtime type → T4 GPU`
2. Conectar el runtime
3. Correr las celdas en orden

In [ ]:
# ── Celda 1: Verificar GPU ────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())
import torch
print('CUDA disponible:', torch.cuda.is_available())
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Celda 2: Montar Google Drive (guarda checkpoints — sobrevive desconexiones) ──
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/qwen35_verilog'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Symlinks para que los checkpoints y datos vayan directo a Drive
os.makedirs('outputs', exist_ok=True)
if not os.path.islink('outputs/checkpoints'):
    os.symlink(f'{DRIVE_DIR}/checkpoints', 'outputs/checkpoints')
    os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

print(f'Drive montado → checkpoints en {DRIVE_DIR}/checkpoints')
print('Si la sesión se corta, los checkpoints estarán en tu Drive.')

In [ ]:
# ── Celda 2: Instalar dependencias ────────────────────────────────────────────
# unsloth: fine-tuning ultra-eficiente para GPUs NVIDIA
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q datasets transformers tqdm rich gitpython
# iverilog para compilar y validar el Verilog generado
!apt-get install -qq iverilog
print('Dependencias instaladas')

In [ ]:
# ── Celda 3: Clonar datos de entrenamiento ────────────────────────────────────
import subprocess
from pathlib import Path

DATA = Path('data/raw')

REPOS = [
    ('verilog-axi',      'https://github.com/alexforencich/verilog-axi.git'),
    ('verilog-uart',     'https://github.com/alexforencich/verilog-uart.git'),
    ('verilog-i2c',      'https://github.com/alexforencich/verilog-i2c.git'),
    ('verilog-ethernet', 'https://github.com/alexforencich/verilog-ethernet.git'),
    ('wb2axip',          'https://github.com/ZipCPU/wb2axip.git'),
    ('spi-master',       'https://github.com/nandland/spi-master.git'),
    ('zipcpu',           'https://github.com/ZipCPU/zipcpu.git'),
    ('basejump-stl',     'https://github.com/bespoke-silicon-group/basejump_stl.git'),
    ('verilog-eval',     'https://github.com/NVlabs/verilog-eval.git'),
    ('RTLLM',            'https://github.com/hkust-zhiyao/RTLLM.git'),
]

for name, url in REPOS:
    dest = DATA / name
    if dest.exists():
        print(f'  skip  {name} (ya existe)')
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth=1', url, str(dest)],
                       capture_output=True)
        print(f'  clone {name}')

# Dataset HuggingFace
from datasets import load_dataset
hf_dest = DATA / 'hf' / 'shailja__Verilog_GitHub'
if not hf_dest.exists():
    print('  Descargando shailja/Verilog_GitHub (~108k ejemplos)...')
    ds = load_dataset('shailja/Verilog_GitHub')
    hf_dest.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(hf_dest))
    print(f'  Guardado: {hf_dest}')
else:
    print('  skip  shailja/Verilog_GitHub (ya existe)')

print('\nDatos listos.')

In [ ]:
# ── Celda 4: Preparar dataset ─────────────────────────────────────────────────
import json, re, random
from pathlib import Path
from tqdm.auto import tqdm

DATA = Path('data/raw')
OUT  = Path('data/processed')
OUT.mkdir(parents=True, exist_ok=True)

# En T4 no se filtra por tokens — el trainer trunca a max_seq_length=4096
# automáticamente sin causar OOM gracias a Flash Attention 2

SYSTEM = """You are an expert RTL designer specialised in synthesisable Verilog and SystemVerilog.
When asked to CREATE a file, output ONLY the complete file content starting with the module declaration.
When asked to MODIFY code, output the complete modified file.
Always use non-blocking assignments (<=) in clocked always blocks.
Filenames follow snake_case (e.g. axi_lite_slave.v)."""

def module_name(src):
    m = re.search(r'\bmodule\s+(\w+)', src)
    return m.group(1) if m else None

def strip_header(src):
    lines, out, in_h = src.splitlines(), [], True
    for line in lines:
        s = line.strip()
        if in_h and (s.startswith('//') or s.startswith('/*') or s == ''):
            continue
        in_h = False
        out.append(line)
    return '\n'.join(out)

TAG_MAP = {
    'uart':'UART serial communication', 'spi':'SPI interface',
    'i2c':'I2C interface', 'axi':'AXI bus interface',
    'axil':'AXI-Lite bus interface', 'fifo':'synchronous FIFO',
    'arb':'round-robin arbiter', 'eth':'Ethernet MAC/PHY interface',
    'wb':'Wishbone bus interface', 'fsm':'finite state machine',
}

def desc(path, mod):
    stem = path.stem.lower()
    for k, v in TAG_MAP.items():
        if k in stem or k in mod.lower(): return v
    return stem.replace('_', ' ')

def make_create(path, src):
    mod = module_name(src)
    if not mod: return None
    return {'messages': [
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': f'Create a file named `{path.name}` implementing a synthesisable {desc(path, mod)} module called `{mod}` in Verilog.'},
        {'role':'assistant', 'content': src.strip()},
    ], 'task': 'create', 'file': path.name}

def make_modify(path, src):
    if 'posedge clk' not in src: return None
    mod = module_name(src)
    if not mod: return None
    modified = src.replace('posedge rst', 'negedge rst_n').replace('if (rst)', 'if (!rst_n)')
    if modified == src: return None
    return {'messages': [
        {'role':'system',    'content': SYSTEM},
        {'role':'user',      'content': f'File `{path.name}`: change reset to active-low (negedge rst_n). Output the complete modified file.\n\n```verilog\n{src.strip()}\n```'},
        {'role':'assistant', 'content': modified.strip()},
    ], 'task': 'modify', 'file': path.name}

samples = []

# Repos clonados
repo_dirs = [d for d in DATA.rglob('*') if d.is_dir() and d.name not in ('verilog-eval', 'RTLLM', 'hf')]
verilog_files = [f for d in repo_dirs for f in d.rglob('*.v')] + \
                [f for d in repo_dirs for f in d.rglob('*.sv')]
print(f'Archivos Verilog encontrados: {len(verilog_files)}')

for f in tqdm(verilog_files, desc='Procesando repos'):
    try:
        src = f.read_text(errors='ignore')
    except Exception:
        continue
    if len(src) < 100:
        continue
    src = strip_header(src)
    for fn in (make_create, make_modify):
        s = fn(f, src)
        if s: samples.append(s)

# HuggingFace dataset
from datasets import load_from_disk
hf_dest = DATA / 'hf' / 'shailja__Verilog_GitHub'
if hf_dest.exists():
    ds = load_from_disk(str(hf_dest))
    split = ds['train'] if 'train' in ds else ds
    print(f'HF dataset: {len(split)} filas raw')
    for row in tqdm(split, desc='HF dataset'):
        inst = row.get('instruction') or row.get('prompt') or ''
        out  = row.get('output') or row.get('completion') or ''
        if not inst or not out: continue
        if not any(k in out for k in ('module ', 'endmodule')): continue
        samples.append({'messages': [
            {'role':'system',    'content': SYSTEM},
            {'role':'user',      'content': inst.strip()},
            {'role':'assistant', 'content': out.strip()},
        ], 'task': 'hf_instruct'})

print(f'\nTotal samples: {len(samples)}')

# Shuffle y split
random.seed(42)
random.shuffle(samples)
n_valid = max(100, int(len(samples) * 0.05))
n_test  = max(100, int(len(samples) * 0.05))
test    = samples[:n_test]
valid   = samples[n_test:n_test + n_valid]
train   = samples[n_test + n_valid:]

for name, data in [('train', train), ('valid', valid), ('test', test)]:
    p = OUT / f'{name}.jsonl'
    with p.open('w') as f:
        for row in data:
            f.write(json.dumps(row) + '\n')
    print(f'  {name}.jsonl: {len(data)} samples')

In [ ]:
# ── Celda 5: QLoRA Fine-Tuning con Unsloth ───────────────────────────────────
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import torch, json
from datasets import Dataset

MAX_SEQ  = 4096   # 9B con 4-bit en T4: 4096 es el máximo seguro
MODEL_ID = 'Qwen/Qwen3.5-9B'

# 9B en bfloat16 = ~18 GB → no cabe en T4
# 9B en 4-bit   = ~6 GB  → cabe con margen para activaciones y LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ,
    dtype          = torch.bfloat16,
    load_in_4bit   = True,   # QLoRA obligatorio en T4 para 9B
)

# LoRA en todas las capas y módulos
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parámetros entrenables: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

train_data = load_jsonl('data/processed/train.jsonl')
valid_data = load_jsonl('data/processed/valid.jsonl')
print(f'Train: {len(train_data)} | Valid: {len(valid_data)}')

def format_sample(sample):
    return tokenizer.apply_chat_template(
        sample['messages'], tokenize=False, add_generation_prompt=False
    )

train_ds = Dataset.from_list([{'text': format_sample(s)} for s in train_data])
valid_ds = Dataset.from_list([{'text': format_sample(s)} for s in valid_data])

# ── EPOCHS ────────────────────────────────────────────────────────────────────
# 1 epoch ≈ 3–5 h   → recomendado para T4 gratis
# 2 epochs ≈ 6–9 h  → Colab Pro o sesión estable
# 3 epochs ≈ 9–14 h → Colab Pro+
NUM_EPOCHS = 1

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = valid_ds,
    args = SFTConfig(
        output_dir                  = 'outputs/checkpoints',
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps                = 50,
        learning_rate               = 2e-5,
        lr_scheduler_type           = 'cosine',
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 25,
        eval_strategy               = 'steps',
        eval_steps                  = 200,
        save_strategy               = 'steps',
        save_steps                  = 200,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        max_seq_length              = MAX_SEQ,
        dataset_text_field          = 'text',
        report_to                   = 'none',
    ),
)

print('Iniciando entrenamiento...')
trainer.train()

In [ ]:
# ── Celda 6: Benchmark rápido con iverilog ────────────────────────────────────
import subprocess, re, tempfile
from pathlib import Path
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)

SYSTEM = """You are an expert RTL designer specialised in synthesisable Verilog and SystemVerilog.
When asked to CREATE a file, output ONLY the complete file content starting with the module declaration."""

def generate(instruction, max_new_tokens=512):
    messages = [
        {'role': 'system',  'content': SYSTEM},
        {'role': 'user',    'content': instruction},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                         temperature=0.1, do_sample=True, use_cache=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

def compile_verilog(code):
    with tempfile.NamedTemporaryFile(suffix='.v', mode='w', delete=False) as f:
        f.write(code)
        fname = f.name
    r = subprocess.run(['iverilog', '-o', '/dev/null', '-g2012', fname],
                       capture_output=True, text=True)
    return r.returncode == 0, r.stderr

# Test con los primeros 20 ejemplos del split de test
import json
test_samples = [json.loads(l) for l in open('data/processed/test.jsonl')][:20]
passed = 0
for s in tqdm(test_samples, desc='Benchmark'):
    user_msg = next(m['content'] for m in s['messages'] if m['role'] == 'user')
    generated = generate(user_msg)
    ok, _ = compile_verilog(generated)
    if ok: passed += 1

print(f'\nCompile pass rate: {passed}/{len(test_samples)} ({passed/len(test_samples)*100:.1f}%)')

In [ ]:
# ── Celda 7: Exportar a GGUF Q8_0 ────────────────────────────────────────────
from pathlib import Path
Path('outputs/gguf').mkdir(parents=True, exist_ok=True)

# Unsloth exporta directamente a GGUF sin necesitar llama.cpp separado
model.save_pretrained_gguf(
    'outputs/gguf/qwen35-verilog',
    tokenizer,
    quantization_method='q8_0',   # Q8_0 = mismo formato que el original en LM Studio
)

import os
gguf_files = list(Path('outputs/gguf').glob('*.gguf'))
for f in gguf_files:
    size_gb = f.stat().st_size / 1e9
    print(f'{f.name}  ({size_gb:.2f} GB)')

In [ ]:
# ── Celda 8: Descargar el GGUF ────────────────────────────────────────────────
# Opción A — descargar directamente al Mac
from google.colab import files
import glob

gguf_path = glob.glob('outputs/gguf/*.gguf')[0]
print(f'Descargando {gguf_path}...')
files.download(gguf_path)

# Opción B — subir a HuggingFace Hub (descomenta si prefieres)
# from huggingface_hub import HfApi
# api = HfApi(token='hf_TU_TOKEN')
# api.upload_file(
#     path_or_fileobj=gguf_path,
#     path_in_repo='qwen35-verilog-q8_0.gguf',
#     repo_id='TU_USUARIO/qwen35-verilog',
#     repo_type='model',
# )

## Instalar en LM Studio

Una vez descargado el `.gguf`, cópialo a la carpeta de modelos de LM Studio:

```bash
cp ~/Downloads/qwen35-verilog-q8_0.gguf \
   ~/Library/Application\ Support/LM\ Studio/models/lmstudio-community/
```

Luego en LM Studio → My Models → aparece listado listo para usar.